# Processing the *E. coli* benchmark tasks

How four built *Escherichia coli* K-12 datasets become benchmark tasks: the dataset,
the genome its records are written against, the records matched to genes of that
genome, the labels, and the pinned train, validation and test splits. Every step is a
function of `torchcell.tasks`, and the last cell is the one import a benchmark script
needs. This file is the source of the notebook
`experiments/050-ecoli-benchmark/notebooks/ecoli_benchmark_processing.ipynb`: a
`# %%` line starts a cell.

Run it from a checkout of the repository, because the cells that rebuild the
sequence-similarity splits read the committed protein clusters.

In [1]:
import os
import warnings

import numpy as np
import pandas as pd

from torchcell.endpoint_http import DEFAULT_URL
from torchcell.sequence.genome.standard import genome_for
from torchcell.tasks import TestSplitLocked, TestSplitWarning, load_task
from torchcell.tasks.ecoli import (
    ECOLI_TASKS,
    build_task,
    match_records,
    open_dataset,
    read_labels,
)
from torchcell.tasks.splits import (
    assign_groups,
    block_groups,
    pair_parts,
    part_sizes,
    random_split,
)

os.environ.setdefault("TC_DATA_URL", DEFAULT_URL)  # download the built dataset stores
os.environ.setdefault("DATA_ROOT", ".")  # stores and genome files go under here
pd.set_option("display.width", 200)

## The four tasks

A task pins one dataset, one genome assembly and one label.

In [2]:
for spec in ECOLI_TASKS.values():
    print(spec.name)
    print(f"  dataset  {spec.dataset_class}  ({spec.kind})")
    print(f"  genome   {spec.assembly_set}")
    print(f"  label    {spec.label}: {spec.n_targets_note}")

ecoli-growth-rate-campos2018
  dataset  GrowthRateCampos2018Dataset  (single_gene)
  genome   ecoli_K12_BW25113_ASM75055v1
  label    fitness: 1 target: maximal growth rate relative to the parent strain
ecoli-morphology-campos2018
  dataset  MorphologyCampos2018Dataset  (single_gene)
  genome   ecoli_K12_BW25113_ASM75055v1
  label    morphology: 15 targets: feature means; 6 are nan for strains imaged without a nucleoid channel
ecoli-gene-interaction-babu2014
  dataset  GeneInteractionBabu2014Dataset  (gene_pair)
  genome   ecoli_K12_MG1655_ASM584v2
  label    gene_interaction: 1 target: eSGA S-score of the pair
ecoli-metabolome-fuhrer2017
  dataset  MetabolomeFuhrer2017Dataset  (single_gene)
  genome   ecoli_K12_BW25113_ASM75055v1
  label    metabolite_level: 7,534 targets: modified z-score of each detected ion


## Step 1: the dataset and its genome

`open_dataset` runs the dataset's loader. `genome_for` reads the assembly the records
name and returns that genome, fetching its files when they are absent.

In [3]:
spec = ECOLI_TASKS["ecoli-metabolome-fuhrer2017"]
dataset = open_dataset(spec)
genome = genome_for(dataset)
print(f"{len(dataset):,} records")
print(
    f"genome {type(genome).__name__} ({genome.ASSEMBLY_SET}), {len(genome.gene_set):,} genes"
)

3,806 records
genome EcoliK12BW25113Genome (ecoli_K12_BW25113_ASM75055v1), 4,303 genes


## Step 2: match records to genes of the genome

A dataset stores each perturbed gene under the identifier its source released (here
Keio JW numbers). `match_records` resolves every one against the genome and decides
whether the record is a benchmark sample: it must perturb the number of genes the
task is about, and each gene must resolve to one gene of the genome that has a
protein sequence. A record that is not a sample keeps its row and its reason.

In [4]:
matched = match_records(dataset, genome, n_genes=1)
print(matched.head(5).to_string(index=False))
print()
print(matched["status"].value_counts().to_string())
print()
print(matched.loc[~matched["eligible"], "exclusion_reason"].value_counts().to_string())

 record_index  n_perturbations stored symbol    locus_tag  status  eligible exclusion_reason
            0                1 JW3210   aaeA BW25113_3241 renamed      True                 
            1                1 JW3209   aaeB BW25113_3240 renamed      True                 
            2                1 JW3212   aaeR BW25113_3243 renamed      True                 
            3                1 JW5541   aaeX BW25113_3242 renamed      True                 
            4                1 JW2804    aas BW25113_2836 renamed      True                 

status
renamed                                               3624
non_gene_feature                                       115
retired                                                 46
non_gene_feature|non_gene_feature                       13
renamed|renamed                                          4
non_gene_feature|non_gene_feature|non_gene_feature       3
non_gene_feature|retired                                 1

exclusion_reason
a g

## Step 3: labels

`read_labels` returns one row per sample and one column per target. A label that maps
names to values (ions here, morphology features for the imaging task) becomes one
column per name, with `nan` where a sample does not hold that name.

In [5]:
samples = matched[matched["eligible"]]
y, targets = read_labels(dataset, spec, samples["record_index"].to_numpy())
print("labels", y.shape, "first targets", targets[:3])

labels (3624, 7534) first targets ['neg_0001', 'neg_0002', 'neg_0003']


`build_task` is steps 1 to 3 in one call. The table counts the result for all four.

In [6]:
built = {name: build_task(name) for name in ECOLI_TASKS}
print(
    pd.DataFrame(
        {
            "records": {n: len(t) + len(t.excluded) for n, t in built.items()},
            "samples": {n: len(t) for n, t in built.items()},
            "excluded": {n: len(t.excluded) for n, t in built.items()},
            "genes": {n: len(set(t.genes.ravel())) for n, t in built.items()},
            "targets": {n: t.y.shape[1] for n, t in built.items()},
        }
    ).to_string()
)

                                 records  samples  excluded  genes  targets
ecoli-growth-rate-campos2018        3664     3573        91   3573        1
ecoli-morphology-campos2018         3664     3573        91   3573       15
ecoli-gene-interaction-babu2014    42705    40658      2047   3726        1
ecoli-metabolome-fuhrer2017         3806     3624       182   3624     7534


## Step 4: splits

A split assigns whole units to train, validation and test in about 80 %, 10 % and
10 %. The pinned split files were written by `build_splits.py` with the functions
below; the cells rebuild one split of each kind and compare it with the pinned
column. For a single-gene task the unit is the gene.

In [7]:
task = load_task("ecoli-growth-rate-campos2018")
genes = [str(g) for g in task.genes[:, 0]]
print(task.split_names)

rebuilt = random_split(genes, seed=0)
print("random_s0  ", part_sizes(rebuilt), (rebuilt == task._splits["random_s0"]).all())

['random_s0', 'random_s1', 'random_s2', 'random_s3', 'random_s4', 'seqid30_s0', 'seqid30_s1', 'seqid30_s2', 'seqid50_s0', 'seqid50_s1', 'seqid50_s2', 'position_s0', 'position_s1', 'position_s2']
random_s0   {'train': 2859, 'val': 357, 'test': 357} True


A sequence-similarity split keeps a protein cluster whole. The clusters are connected
components of an all-against-all MMseqs2 search at 30 % (or 50 %) identity over 80 %
of both sequences (`cluster_proteins.py`), so a test protein has no reported hit to a
training protein at or above the threshold.

In [8]:
clusters = pd.read_csv(
    f"experiments/050-ecoli-benchmark/results/clusters/{task.spec.assembly_set}.seqid30.tsv",
    sep="\t",
    dtype=str,
)
cluster_of = dict(zip(clusters["locus_tag"], clusters["cluster"]))
rebuilt = assign_groups([cluster_of[g] for g in genes], seed=0)
print("seqid30_s0 ", part_sizes(rebuilt), (rebuilt == task._splits["seqid30_s0"]).all())
parts_of_cluster = pd.Series(rebuilt).groupby([cluster_of[g] for g in genes]).nunique()
print("clusters divided between parts:", int((parts_of_cluster > 1).sum()))

seqid30_s0  {'train': 2859, 'val': 357, 'test': 357} True
clusters divided between parts: 0


A position split keeps a chromosome block whole (40 contiguous blocks of equal gene
count), so the genes of an operon stay on one side.

In [9]:
starts = [int(task.genome[g].start) for g in sorted(set(genes))]
block_of = dict(zip(sorted(set(genes)), block_groups(starts, n_blocks=40)))
rebuilt = assign_groups([block_of[g] for g in genes], seed=0)
print(
    "position_s0", part_sizes(rebuilt), (rebuilt == task._splits["position_s0"]).all()
)

position_s0 {'train': 2859, 'val': 445, 'test': 269} True


For the gene-pair task a random split keeps the two orders of a pair together. A
gene-holdout split divides the genes first: a pair is test when either gene is a
test gene, validation when neither is test and either is validation, and train when
both are training genes. Holding genes out by cluster or by block works the same way.

In [10]:
pairs = load_task("ecoli-gene-interaction-babu2014")
a = [str(g) for g in pairs.genes[:, 0]]
b = [str(g) for g in pairs.genes[:, 1]]
print(pairs.split_names)

pair_genes = sorted(set(a) | set(b))
gene_part = dict(zip(pair_genes, assign_groups(pair_genes, seed=0)))
rebuilt = pair_parts([gene_part[g] for g in a], [gene_part[g] for g in b])
print(
    "gene_holdout_s0",
    part_sizes(rebuilt),
    (rebuilt == pairs._splits["gene_holdout_s0"]).all(),
)

['random_s0', 'random_s1', 'random_s2', 'random_s3', 'random_s4', 'gene_holdout_s0', 'gene_holdout_s1', 'gene_holdout_s2', 'seqid30_gene_holdout_s0', 'seqid30_gene_holdout_s1', 'seqid30_gene_holdout_s2', 'seqid50_gene_holdout_s0', 'seqid50_gene_holdout_s1', 'seqid50_gene_holdout_s2', 'position_gene_holdout_s0', 'position_gene_holdout_s1', 'position_gene_holdout_s2']
gene_holdout_s0 {'train': 25848, 'val': 8642, 'test': 6168} True


## The short import

Everything above is what `load_task` does. A benchmark script needs this import and
nothing else from the processing, which is also what a submission is checked for.

In [11]:
task = load_task("ecoli-growth-rate-campos2018")
split = task.split("seqid30_s0")
print(f"{len(task):,} samples, {task.y.shape[1]} target, sizes {split.sizes()}")
print(
    "first training genes",
    task.genes[split.train[:3], 0],
    "labels",
    task.y[split.train[:3], 0],
)

3,573 samples, 1 target, sizes {'train': 2859, 'val': 357, 'test': 357}
first training genes ['BW25113_3934' 'BW25113_1334' 'BW25113_0676'] labels [0.8306286  0.9603691  0.98156133]


The test part is locked. Reading it raises, and the message says what to do instead.

In [12]:
try:
    split.test
except TestSplitLocked as error:
    print(error)

the test part of ecoli-growth-rate-campos2018/seqid30_s0 is locked. Select models and hyperparameters on .val. For the one final evaluation call split.unlock_test(reason='...').


Model selection uses validation. As an example, the constant prediction of the
training mean, scored on validation with the task's metrics.

In [13]:
prediction = np.full((len(split.val), 1), task.y[split.train].mean())
print({k: round(v, 4) for k, v in task.score(task.y[split.val], prediction).items()})

{'pearson': nan, 'spearman': nan, 'r2': -0.0002, 'mse': 0.0063, 'mae': 0.0496}


`unlock_test` returns the test indices and warns every time. It is for one final
evaluation of a model chosen on validation, and the reason is part of the call.

In [14]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always", TestSplitWarning)
    test = split.unlock_test(reason="showing the warning in the processing notebook")
print(len(test), "test samples")
print(caught[0].category.__name__ + ":", caught[0].message)

357 test samples
